# Notebook 03a — What Do the LoRA Knobs Actually Do?

**Varies:** `02_instruction_finetuning_lora.ipynb`. The `a` suffix means *not a pipeline stage* —
this notebook trains 16 adapters, measures them, and throws them all away. Nothing downstream
consumes its output.
**Data:** the same 800 / 200 PubMedQA split notebook 02 uses, rebuilt from the same seed
**Hardware:** A100 40GB · **Runtime:** ~90 minutes · (a free T4 runs it in ~2.5 h — see section 8)

---

## Why this notebook exists

Notebooks 01, 02 and 03 all train at exactly one configuration: `r=16`, `lora_alpha=32`, all seven
projections, `lr=2e-4`. Those values are defensible defaults, but nothing in this repo has ever
checked them — and `docs/concepts.md` makes three claims about them that no cell tests:

| the claim | §3 says | the arms that test it |
|---|---|---|
| **`target_modules` matters more than `r`** | "If domain adaptation underdelivers, check `target_modules` before touching `r`." | `attn-only` against the two rank arms |
| **`alpha = 2r` decouples capacity from step size** | "raising `r` adds capacity without changing effective step size" | the `alpha/r` axis against the `r` axis |
| **rank 64+ is rarely justified on small corpora** | "Larger — rarely justified on small corpora" | `r=64` at three learning rates |

So this is not a sweep for the sake of having one. It is the experiment that audits the
documentation.

## The design, and the one thing that makes it honest

A full grid over the four axes — 3 ranks × 3 scalings × 2 target sets × 3 learning rates — is
**54 runs and about nine hours**. Worse, it would be nine hours spent measuring differences smaller
than the measurement error. So instead:

```
Block A   one factor at a time, 9 arms      each arm = notebook 02's config with ONE key changed
Block B   the rank x lr grid,   4 arms      the one interaction Block A structurally cannot see
Block C   replicate the winners, 3 arms     does the best arm stay best when you only change the seed?
```

**The accuracy of a 1B model on 200 records has a standard error of about 3.5 points.** A naive 95%
interval spans ±7. Most of the deltas this notebook produces will be smaller than that, and a table
of 16 accuracies with no error estimate would be a **worse** artifact than no sweep at all, because
it would invite conclusions the data cannot support.

So Block A spends one of its nine arms retraining the baseline with **nothing changed but the
random seed**, and Block C spends three more the same way. Four paired seed-deltas give a measured
noise floor, and every other number in this notebook is reported against it.

Expect most arms to land inside that floor. **That is the finding**, not a failure of the sweep.

**Set your runtime to a GPU first:** Runtime → Change runtime type → A100 (or any GPU).

## 1. Runtime and dependencies

Identical to notebooks 01–03.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "No GPU found — set Runtime > Change runtime type > GPU")

In [ ]:
%pip install -q "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0"
# Colab preinstalls torchao 0.10.x. peft's optional torchao integration RAISES rather than
# degrading gracefully when it finds a version below its 0.16.0 minimum, and the check fires
# deep inside get_peft_model() — which this notebook calls sixteen times.
%pip uninstall -q -y torchao
print("\nRestart the runtime if Colab asks you to, then run this cell again and continue.")

In [ ]:
import gc, json, math, random, re, time
from pathlib import Path
from collections import Counter

import torch
import transformers
import peft
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > GPU."

# bfloat16 needs Ampere (sm_80) or later; a T4 is Turing (sm_75). Checking compute
# capability directly, because torch.cuda.is_bf16_supported() has returned True on Turing.
SUPPORTS_BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"

MODEL_ID = "unsloth/Llama-3.2-1B"

print(f"transformers {transformers.__version__} | GPU {torch.cuda.get_device_name(0)}")
print(f"compute dtype {DTYPE} | from_pretrained keyword {DTYPE_KW!r}")

# Fail fast on the torchao clash, rather than eight cells from now inside get_peft_model().
import importlib.metadata as _md


def _ver(s):
    return tuple(int(p) for p in s.split("+")[0].split(".")[:3] if p.isdigit())


try:
    _ta = _md.version("torchao")
    if _ver(_ta) < (0, 16, 0):
        raise RuntimeError(
            f"torchao {_ta} is too old for peft {peft.__version__} (needs >= 0.16.0).\n"
            "Nothing in this notebook uses torchao. Fix it with:\n"
            "    !pip uninstall -y torchao\n"
            "then Runtime > Restart session, then Runtime > Run all."
        )
    print(f"torchao      {_ta} (compatible)")
except _md.PackageNotFoundError:
    print("torchao      absent - fine, nothing here uses it")

## 2. Find the stage-1 adapter

Every arm trains on the **same** domain-adapted base that notebook 02 uses, so the sweep measures
the stage-2 configuration and nothing else. Notebook 01 saved that adapter to
`MyDrive/finetuning-demo/stage1-domain-lora`.

If you skipped notebook 01, set `SKIP_STAGE1 = True`. The sweep still works — every arm just starts
from the raw base instead — but the baseline arm is then no longer notebook 02's configuration.

In [ ]:
SKIP_STAGE1 = False

STAGE1_DIR = None
if not SKIP_STAGE1:
    candidates = [
        Path("/content/drive/MyDrive/finetuning-demo/stage1-domain-lora"),
        Path("/content/outputs/stage1-domain-lora"),
    ]
    if not any(c.exists() for c in candidates):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except ImportError:
            pass
    STAGE1_DIR = next((c for c in candidates if (c / "adapter_config.json").exists()), None)

if STAGE1_DIR is None:
    print("No stage-1 adapter found. Either run notebook 01 first, or set SKIP_STAGE1 = True.")
else:
    print(f"Found stage-1 adapter: {STAGE1_DIR}\n")
    cfg = json.loads((STAGE1_DIR / "adapter_config.json").read_text())
    for key in ["base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules"]:
        print(f"  {key:26s} {cfg.get(key)}")

## 3. Rebuild the domain-adapted base by merging

Identical to notebook 02 section 3: load the pristine base, attach the stage-1 adapter with
`PeftModel.from_pretrained`, `merge_and_unload()` to fold `(α/r)·BA` into `W`, and **verify** the
weights actually changed rather than assuming it.

One naming difference worth noting: the merged model is called `base` here, not `model`. Sixteen
arms each bind their own `model`, and `base` is the thing they all start from.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"          # training, not generation
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS must stay distinct"

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
base.config.pad_token_id = tokenizer.pad_token_id

# Snapshot a weight the stage-1 adapter targeted, so we can prove the merge changed it.
PROBE_WEIGHT = "model.layers.8.mlp.down_proj.weight"
before = dict(base.named_parameters())[PROBE_WEIGHT].detach().clone()
print(f"Loaded pristine {MODEL_ID}")

In [ ]:
if STAGE1_DIR is not None:
    base = PeftModel.from_pretrained(base, STAGE1_DIR)
    print(f"Attached stage-1 adapter ({type(base).__name__})")
    base = base.merge_and_unload()
    print(f"Merged and unwrapped   ({type(base).__name__})")

    after = dict(base.named_parameters())[PROBE_WEIGHT].detach()
    drift = (after.float() - before.float()).abs().max().item()
    changed = not torch.equal(after, before)

    print(f"\nVerification on {PROBE_WEIGHT}")
    print(f"  weights changed : {changed}")
    print(f"  max |delta|     : {drift:.6f}")
    assert changed, (
        "The merge did not change any weights. The adapter was not applied — this is exactly "
        "the failure mode that AutoModelForCausalLM.from_pretrained(adapter_dir) produces."
    )
    print("\n  -> stage-1 knowledge is now inside the base weights")
else:
    print("SKIP_STAGE1 is set: the sweep runs on the raw base model.")

del before, after
gc.collect(); torch.cuda.empty_cache()
print(f"\nGPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### The snapshot every arm restores from

`get_peft_model()` does not copy the model — it walks the module tree and **replaces**
`nn.Linear` with `lora.Linear` in place. Calling it sixteen times on one object would stack
sixteen adapters, and every arm after the first would be measuring something other than what its
name says.

So we keep a pristine copy of the merged weights on the CPU and restore from it between arms. Two
details that would otherwise cost host RAM and a debugging session:

- Llama-3.2-1B **ties** `lm_head.weight` to `model.embed_tokens.weight`, so `state_dict()` returns
  the same tensor under two keys. Copying naively makes two independent 0.53 GB CPU copies.
  De-duplicating by storage pointer keeps one.
- The probe statistics printed below are the fingerprint of *this* starting point. bf16 and fp16
  round these weights differently, and the stage-1 merge arithmetic runs in whichever one your GPU
  has — so two people on different hardware can compare one line and see how far apart they start.

In [ ]:
BASE_SNAPSHOT, _by_storage = {}, {}
for k, v in base.state_dict().items():
    ptr = (v.device, v.data_ptr())
    if ptr not in _by_storage:
        _by_storage[ptr] = v.detach().to("cpu", copy=True)
    BASE_SNAPSHOT[k] = _by_storage[ptr]

snap_gb = sum(t.numel() * t.element_size() for t in _by_storage.values()) / 1e9
tied = len(BASE_SNAPSHOT) - len(_by_storage)
print(f"CPU snapshot: {len(BASE_SNAPSHOT)} keys, {len(_by_storage)} distinct tensors "
      f"({tied} tied), {snap_gb:.2f} GB host RAM")
del _by_storage

_p = BASE_SNAPSHOT[PROBE_WEIGHT].float()
print(f"\nfingerprint of the merged base — {PROBE_WEIGHT}")
print(f"  dtype {BASE_SNAPSHOT[PROBE_WEIGHT].dtype}   mean {_p.mean():+.6e}   "
      f"std {_p.std():.6e}   absmax {_p.abs().max():.6e}")
del _p

# Set by run_arm(). fresh_base() uses it to detach the previous arm's adapter — including when
# that arm died halfway through and left one attached.
ACTIVE_PEFT = None

## 4. The data — notebook 02's split, rebuilt

Not a new split. The same constants and the same shuffle recipe produce the same 800 / 200
partition, which is what lets the baseline arm here be compared against notebook 02's.

The order of operations in the split cell is load-bearing: sort by decision class, sort within each
class by `pubid`, *then* shuffle with a seeded RNG. Iterating a dict or trusting the dataset's own
order would silently give a different 200 records.

Notebook 02 also asserts that the two stages share no articles. That assert is not repeated here —
it costs a 61,249-row download to check something notebooks 02 and 03 already establish, and this
notebook's eval set is theirs.

In [ ]:
from datasets import load_dataset

# --- constants mirrored in scripts/prepare_data.py and notebooks 01-03 ---
DATASET = "qiaojin/PubMedQA"
SEED = 20260815
EVAL_FRACTION = 0.20      # -> 800 train / 200 eval

TASK = (
    "Answer the research question using only the abstract provided. "
    'Begin your reply with "Answer:" followed by yes, no, or maybe, '
    "then justify it in one or two sentences."
)

labeled = load_dataset(DATASET, "pqa_labeled", split="train")
print(labeled)

In [ ]:
def join_sections(context) -> str:
    """Render an abstract's sections as `LABEL: text` blocks."""
    labels = context.get("labels") or []
    parts = []
    for i, text in enumerate(context["contexts"]):
        text = " ".join(text.split())
        if not text:
            continue
        label = (labels[i] if i < len(labels) else "").strip()
        parts.append(f"{label}: {text}" if label else text)
    return "\n\n".join(parts)


records = []
for row in labeled:
    abstract = join_sections(row["context"])
    if not abstract:
        continue
    records.append({
        "instruction": f'{TASK}\n\nQuestion: {row["question"].strip()}',
        "input": abstract,
        "output": f'Answer: {row["final_decision"]}\n\n{" ".join(row["long_answer"].split())}',
        "decision": row["final_decision"],
        "pubid": row["pubid"],
    })

print(f"{len(records)} instruction records built")

In [ ]:
rng = random.Random(SEED + 1)
by_decision = {}
for r in records:
    by_decision.setdefault(r["decision"], []).append(r)

train_records, eval_records = [], []
for decision in sorted(by_decision):
    rows = sorted(by_decision[decision], key=lambda r: r["pubid"])
    rng.shuffle(rows)
    n_eval = round(len(rows) * EVAL_FRACTION)
    eval_records.extend(rows[:n_eval])
    train_records.extend(rows[n_eval:])
rng.shuffle(train_records); rng.shuffle(eval_records)

print(f"train {len(train_records)}  {dict(Counter(r['decision'] for r in train_records))}")
print(f"eval  {len(eval_records)}  {dict(Counter(r['decision'] for r in eval_records))}")
print(f"\nfirst five eval pubids: {[r['pubid'] for r in eval_records[:5]]}")
print("Those should match notebook 02's. If they don't, the two notebooks are not comparable.")

## 5. Prompt, masking, collator

All three copied unchanged from notebook 02 — the Alpaca template, the `-100` completion-only
masking, and the dynamic-padding collator. Notebook 02 sections 5–7 explain why each is built the
way it is; this notebook holds them fixed so that the only thing varying between arms is the LoRA
configuration.

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction describing a task, paired with input providing further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PROMPT_NO_INPUT = (
    "Below is an instruction describing a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)


def build_prompt(record) -> str:
    """The part the model reads but is NOT trained to produce."""
    template = PROMPT_WITH_INPUT if record["input"].strip() else PROMPT_NO_INPUT
    return template.format(instruction=record["instruction"].strip(),
                           input=record["input"].strip())

In [ ]:
MAX_LEN = 896   # p99 of this dataset is ~714 tokens; 896 truncates nothing


def tokenize_record(record):
    """Tokenize one record, masking the prompt out of the loss."""
    prompt = build_prompt(record)
    answer = record["output"].strip()

    # add_special_tokens=True puts BOS at the front of the prompt only.
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    answer_ids = answer_ids + [tokenizer.eos_token_id]   # <-- teaches the model to stop

    input_ids = (prompt_ids + answer_ids)[:MAX_LEN]
    # -100 over the prompt span, real ids over the answer span.
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]

    return {"input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels}


train_tok = [tokenize_record(r) for r in train_records]
eval_tok = [tokenize_record(r) for r in eval_records]

supervised = sum(sum(1 for t in r["labels"] if t != -100) for r in train_tok)
total_tok = sum(len(r["labels"]) for r in train_tok)
print(f"supervised tokens: {supervised:,} / {total_tok:,} = {supervised/total_tok:.1%}")

truncated = sum(1 for r in train_tok + eval_tok if len(r["input_ids"]) >= MAX_LEN)
assert truncated == 0, f"{truncated} records hit MAX_LEN"
print(f"records truncated at MAX_LEN={MAX_LEN}: {truncated}")

In [ ]:
from datasets import Dataset


def collate(features):
    """Dynamic padding. Note the three different pad values."""
    longest = max(len(f["input_ids"]) for f in features)

    def pad(seq, value):
        return seq + [value] * (longest - len(seq))

    return {
        "input_ids": torch.tensor([pad(f["input_ids"], tokenizer.pad_token_id) for f in features]),
        "attention_mask": torch.tensor([pad(f["attention_mask"], 0) for f in features]),
        "labels": torch.tensor([pad(f["labels"], -100) for f in features]),
    }


demo = collate([train_tok[0], train_tok[1]])
print({k: tuple(v.shape) for k, v in demo.items()})

## 6. `QUICK` — check the harness before spending the GPU hour

Sixteen arms is ninety minutes. Finding a typo in the results table at minute eighty-nine is
avoidable, so this one toggle shrinks every arm to a few seconds and runs the whole notebook
end to end in about six minutes.

Three things make the smoke test worth trusting:

- **The scored subset is a prefix of the eval set**, so `eval_loss` and `accuracy` describe the same
  records in both modes, and `MAJORITY_BASELINE` is recomputed rather than inherited from the
  200-record split.
- **`warmup_steps` scales down.** With 64 records and one epoch there are 8 optimizer steps, and
  notebook 02's `warmup_steps=20` would mean the learning rate never leaves warmup — the smoke test
  would exercise a completely different schedule from the real run. The assert below is the guard.
- **QUICK results are written to a different file** *and* recorded inside the run config, so a smoke
  run cannot be mistaken for, or merged into, real numbers.

In [ ]:
QUICK = True     # <-- THE ONE TOGGLE.  True: ~6 min, numbers meaningless.  False: the real run.

if QUICK:
    N_TRAIN, N_EVAL, EPOCHS, WARMUP_STEPS, LOGGING_STEPS = 64, 32, 1, 2, 1
else:
    N_TRAIN, N_EVAL = len(train_records), len(eval_records)
    EPOCHS, WARMUP_STEPS, LOGGING_STEPS = 3, 20, 10

train_ds = Dataset.from_list(train_tok[:N_TRAIN])
eval_ds = Dataset.from_list(eval_tok[:N_EVAL])
score_records = eval_records[:N_EVAL]     # same prefix as eval_ds, so loss and accuracy agree
gold_scored = [r["decision"] for r in score_records]

_counts = Counter(gold_scored)
MAJORITY_LABEL, _n = _counts.most_common(1)[0]
MAJORITY_BASELINE = _n / len(score_records)

STEPS_PER_ARM = (N_TRAIN // 8) * EPOCHS          # micro-batch 2 x accum 4 = effective 8
assert WARMUP_STEPS < STEPS_PER_ARM, (
    f"warmup_steps={WARMUP_STEPS} >= {STEPS_PER_ARM} total optimizer steps: the learning rate "
    "would never leave warmup, so the smoke test would exercise a different schedule than the "
    "real run and tell you nothing about it.")

print(f"{'QUICK SMOKE TEST — numbers are meaningless' if QUICK else 'FULL RUN'}")
print(f"  train {N_TRAIN}  ·  eval/score {N_EVAL}  ·  {EPOCHS} epoch(s)  ·  "
      f"{STEPS_PER_ARM} optimizer steps/arm  ·  warmup {WARMUP_STEPS}")
print(f"  majority-class baseline on the scored subset: {MAJORITY_BASELINE:.1%} "
      f"(always '{MAJORITY_LABEL}')   {dict(_counts)}")

## 7. Identical starting weights for every arm

This is the cell that decides whether the whole notebook means anything. Sixteen arms share one
model object, and there are two ways that goes wrong.

**The one people expect** is drift in the base weights. It does not actually happen:
`lora.Linear` keeps a *reference* to the original `nn.Linear` in `self.base_layer` and its forward
is out-of-place — `base_layer(x) + scaling · B(A(x))`. And the fp32 upcast in `run_arm` cannot
reach a base weight, because peft has already set `requires_grad=False` on every non-LoRA
parameter before that loop runs. The `load_state_dict` below is insurance, not the mechanism.

**The one that would actually ruin the sweep** is `merge_and_unload()`. It folds the *current*
arm's adapter into the weights the *next* arm starts from — contamination that accumulates over
sixteen arms and that no per-arm metric would reveal, because every arm still trains, still
converges, and still reports a plausible number. The rule is simple: `unload()` inside the loop,
never `merge_and_unload()`.

`unload()` also handles the case where an arm died halfway and left an adapter attached, so
re-running the sweep cell after a crash just works.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

REBUILD_FROM_DISK = False   # escape hatch: if a future peft renames unload(), flip this and
                            # fresh_base() reloads + re-merges from disk instead (~15 s/arm).


def fresh_base():
    """Return the merged stage-1 model in the exact state every arm must start from."""
    global base, ACTIVE_PEFT

    if ACTIVE_PEFT is not None:
        # unload(), NOT merge_and_unload(). merge_and_unload() would fold this arm's adapter into
        # the weights the next arm has to start from -- silent, cumulative contamination.
        base = ACTIVE_PEFT.unload()
        ACTIVE_PEFT = None

    if REBUILD_FROM_DISK:
        del base
        gc.collect(); torch.cuda.empty_cache()
        base = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
        base.config.pad_token_id = tokenizer.pad_token_id
        if STAGE1_DIR is not None:
            base = PeftModel.from_pretrained(base, STAGE1_DIR).merge_and_unload()
    else:
        # assign=False copies INTO the existing CUDA parameters. assign=True would REPLACE them
        # with these CPU tensors, and the next forward would die on a device mismatch.
        base.load_state_dict(BASE_SNAPSHOT, strict=True, assign=False)

    # --- assert, don't trust. Three checks, each catching a different failure. ---
    stragglers = [n for n, _ in base.named_modules() if "lora" in n.lower()]
    assert not stragglers, (
        f"{len(stragglers)} LoRA modules survived unload(), e.g. {stragglers[:3]}. A second "
        "get_peft_model() would stack adapters and every number after this point would be wrong.")
    assert type(base).__name__ == "LlamaForCausalLM", \
        f"expected a bare base model, got {type(base).__name__}"
    assert torch.equal(dict(base.named_parameters())[PROBE_WEIGHT].detach().cpu(),
                       BASE_SNAPSHOT[PROBE_WEIGHT]), \
        "the merged base weights drifted between arms — the arms are no longer comparable"

    # After the first unload() the base params are still frozen, whereas on arm 1 they arrive from
    # from_pretrained() unfrozen. get_peft_model re-freezes either way, but arm 1 should not be
    # structurally different from the other fifteen.
    for p in base.parameters():
        p.requires_grad_(False)
    base.config.use_cache = False
    tokenizer.padding_side = "right"     # score() leaves it "left"
    return base

### Prove it, before spending ninety minutes on it

Attach a throwaway adapter, push its `lora_B` off zero so the model genuinely computes something
different, then check that `fresh_base()` puts everything back.

The comparison is on **logits**, not on a single weight tensor. A one-parameter `torch.equal` would
pass even if a module came back in the wrong dtype, a hook survived, or something landed on the
wrong device. A bit-exact forward pass would not.

In [ ]:
@torch.no_grad()
def _fingerprint(m):
    m.eval()
    b = collate([train_tok[0], train_tok[1]])
    return m(input_ids=b["input_ids"].to("cuda"),
             attention_mask=b["attention_mask"].to("cuda")).logits.detach().clone()


_before = _fingerprint(base)

_tmp = get_peft_model(base, LoraConfig(task_type=TaskType.CAUSAL_LM, r=4, lora_alpha=8,
                                       lora_dropout=0.0, bias="none", target_modules=["q_proj"]))
ACTIVE_PEFT = _tmp
for _n, _p in _tmp.named_parameters():
    if "lora_B" in _n:            # B is initialised to zero; perturb it so the adapter DOES something
        _p.data.normal_(0, 0.02)

_perturbed = _fingerprint(_tmp)
_delta = (_perturbed.float() - _before.float()).abs().max().item()
print(f"throwaway adapter attached and perturbed: max |delta logits| = {_delta:.4f}   (must be > 0)")
assert _delta > 0, "the throwaway adapter changed nothing — this test would prove nothing"

base = fresh_base()
_after = _fingerprint(base)
_drift = (_after.float() - _before.float()).abs().max().item()
print(f"after fresh_base(): {type(base).__name__}, max |delta logits| vs the original = {_drift:.6e}")
assert _drift == 0.0, (
    "fresh_base() did not restore the model exactly. Either the restore is broken, or this GPU's "
    "forward pass is nondeterministic — in which case no comparison in this notebook is exact, "
    "and you should say so when you report the numbers.")

del _tmp, _before, _perturbed, _after
gc.collect(); torch.cuda.empty_cache()
print(f"\nRound-trip verified. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 8. What changes, and what is held fixed

### One factor at a time

Every Block A arm is notebook 02's configuration with **exactly one key overridden**. That is
weaker than a full grid in one specific way — it cannot see interactions — and stronger in
another: when an arm moves, you know what moved it.

The arms are ordered by how much you lose if the session dies. `baseline` and `baseline-s43` come
first, because together they are this notebook's headline claim; without both, there is no noise
floor and nothing else can be interpreted.

### The interaction worth buying

The folklore claim about rank is that **higher rank needs a lower learning rate**. If that is true
and we only ever test rank at `lr=2e-4`, the `r=64` arm looks bad for a reason that has nothing to
do with rank. So Block B fills in the rest of the 3×3 rank × lr grid. Block A already supplies five
of the nine cells, so this costs four arms rather than nine — and the de-duplication is *computed
and asserted*, not hand-maintained, so a later edit to Block A cannot silently leave a hole in the
grid or pay for the same run twice.

### What is deliberately not tuned

Batch sizes stay at notebook 02's values — micro-batch 2, accumulation 4, eval batch 2, scoring
batch 8 — even though an A100 would happily run much larger ones.

This is not an oversight, and it is worth being precise about why. `Trainer` reports evaluation
loss as an **unweighted mean over batches** of each batch's mean token loss. Under dynamic padding,
regrouping the same records into different batches therefore produces a different `eval_loss` **for
an identical model**. Greedy generation has the same property more weakly: batch composition
changes the padding, which perturbs near-ties in the argmax, which can flip a parsed decision.

So every batch knob touches a reported number. There is no free utilisation win here — only a way
to make sixteen arms quietly incomparable.

In [ ]:
ALL_LINEAR = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
ATTN_ONLY = ["q_proj", "k_proj", "v_proj", "o_proj"]

BASE_SEED, REPLICATE_SEED, SECOND_REPLICATE_SEED = 42, 43, 44


def arm(name, axis, r=16, alpha=32, targets=ALL_LINEAR, lr=2e-4, seed=BASE_SEED, note=""):
    """One training configuration. The defaults ARE notebook 02's configuration."""
    return {"name": name, "axis": axis, "r": r, "alpha": alpha,
            "targets": list(targets), "lr": lr, "seed": seed, "note": note}


def train_key(a):
    """Everything that determines the trained model. Two arms sharing a key would train the
    identical run twice — which is what the de-duplication below exists to prevent."""
    return (a["r"], a["alpha"], tuple(a["targets"]), a["lr"], a["seed"])


def lr_tag(lr):
    return f"{lr:.0e}".replace("e-0", "e-")      # 5e-05 -> 5e-5, stable as a dict key


# --- Block A: one factor at a time ----------------------------------------------------------
BLOCK_A = [
    arm("baseline",     "baseline", note="notebook 02's exact configuration"),
    arm("baseline-s43", "noise",    seed=REPLICATE_SEED, note="identical config, seed only"),
    arm("attn-only",    "targets",  targets=ATTN_ONLY, note="drop the three MLP projections"),
    arm("lr-5e-4",      "lr",       lr=5e-4),
    arm("lr-5e-5",      "lr",       lr=5e-5),
    arm("r-64",         "rank",     r=64, alpha=128, note="alpha/r held at 2"),
    arm("r-8",          "rank",     r=8,  alpha=16,  note="alpha/r held at 2"),
    arm("alpha-r-4",    "scaling",  alpha=64, note="alpha/r = 4"),
    arm("alpha-r-1",    "scaling",  alpha=16, note="alpha/r = 1"),
]

# --- Block B: the rank x lr grid ------------------------------------------------------------
RUN_GRID = True
GRID_R, GRID_LR = [8, 16, 64], [5e-5, 2e-4, 5e-4]

_seen = {train_key(a) for a in BLOCK_A}
BLOCK_B, BORROWED = [], []
for _r in GRID_R:
    for _lr in GRID_LR:
        _cell = arm(f"grid-r{_r}-{lr_tag(_lr)}", "rank x lr", r=_r, alpha=2 * _r, lr=_lr)
        if train_key(_cell) in _seen:
            BORROWED.append((_r, _lr))          # Block A already ran this exact configuration
            continue
        _seen.add(train_key(_cell))
        BLOCK_B.append(_cell)

assert len(BORROWED) == 5 and len(BLOCK_B) == 4, (
    f"Block A should supply 5 of the 9 grid cells, leaving 4 new arms; got {len(BORROWED)} "
    f"borrowed and {len(BLOCK_B)} new. Block A's r / alpha / lr values have drifted off the grid.")

ARMS = BLOCK_A + (BLOCK_B if RUN_GRID else [])
print(f"Block A: {len(BLOCK_A)} arms   Block B: {len(BLOCK_B)} new arms "
      f"({len(BORROWED)} of the 9 grid cells borrowed from Block A)")

In [ ]:
EST_MIN_PER_ARM = 0.35 if QUICK else 5.0

print(f"{'arm':16s}{'axis':11s}{'r':>4s}{'alpha':>7s}{'a/r':>6s}{'tgt':>5s}{'lr':>9s}{'seed':>6s}  note")
print("-" * 104)
for a in ARMS:
    print(f"{a['name']:16s}{a['axis']:11s}{a['r']:>4d}{a['alpha']:>7d}{a['alpha']/a['r']:>6g}"
          f"{len(a['targets']):>5d}{a['lr']:>9.0e}{a['seed']:>6d}  {a['note']}")
print("-" * 104)
print(f"{len(ARMS)} arms x ~{EST_MIN_PER_ARM:.1f} min = ~{len(ARMS) * EST_MIN_PER_ARM:.0f} min "
      f"for blocks A and B, plus block C below.")
print("\nHeld fixed across every arm: micro-batch 2, accumulation 4 (effective 8), eval batch 2,")
print("scoring batch 8, lora_dropout 0.05, cosine schedule, weight decay 0.01, max grad norm 1.0,")
print(f"{EPOCHS} epoch(s), and the same {N_TRAIN}/{N_EVAL} records in the same order.")

## 9. Scoring

Copied unchanged from notebook 02 section 11: greedy left-padded batched generation, then a regex
for the leading `Answer: <yes|no|maybe>`. An unparseable answer is scored as wrong, because an
unparseable answer is useless downstream — and the parse rate is reported separately so you can see
*why* an arm scored badly.

Greedy decoding matters more here than it did in notebook 02. Sampling would add a second source of
run-to-run variance on top of the seed, and separating those two is the entire point of the noise
floor.

In [ ]:
DECISION_RE = re.compile(r"answer\s*:?\s*\b(yes|no|maybe)\b", re.IGNORECASE)


def parse_decision(text):
    """Notebook 02's grading regex. None means unparseable, which is scored as wrong."""
    hit = DECISION_RE.search(text)
    return hit.group(1).lower() if hit else None


@torch.no_grad()
def generate_batch(m, batch, max_new_tokens=120):
    """Greedy, left-padded batched generation. Deterministic so runs are comparable."""
    m.eval()
    m.config.use_cache = True
    prompts = [build_prompt(r) for r in batch]
    # Left padding for generation: every sequence then ends at the same position,
    # which is where the model continues from. (Training used right padding.)
    tokenizer.padding_side = "left"
    enc = tokenizer(prompts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_LEN).to("cuda")
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                     eos_token_id=tokenizer.eos_token_id,
                     pad_token_id=tokenizer.pad_token_id)
    tokenizer.padding_side = "right"
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]


def score(m, dataset, label, batch_size=8):
    """Decision accuracy + parse rate over the held-out set."""
    texts = []
    for i in range(0, len(dataset), batch_size):
        texts.extend(generate_batch(m, dataset[i:i + batch_size]))
        print(f"\r  scoring {label}: {min(i + batch_size, len(dataset))}/{len(dataset)}", end="")
    print()
    preds = [parse_decision(t) for t in texts]
    correct = sum(1 for p, r in zip(preds, dataset) if p == r["decision"])
    parsed = sum(1 for p in preds if p is not None)
    return {"accuracy": correct / len(dataset), "parse_rate": parsed / len(dataset),
            "preds": preds, "texts": texts}

## 10. One arm

### The line that makes one-factor-at-a-time true

`get_peft_model` draws `lora_A` from a `kaiming_uniform_` on the **global** torch RNG.
`Trainer.__init__` calls `set_seed(args.seed)` — but that happens *afterwards*, when the adapter
has already been initialised.

Left alone, arm *k*'s initialisation is therefore a function of everything arm *k−1* did: its
training steps, its dropout draws, its 200 generations. The eight `seed=42` arms would differ in
the knob under test **and** in where they started from, and "the only thing that changed is the
knob" would be false. The explicit `set_seed` immediately before `get_peft_model` is what makes the
claim in this notebook's title honest.

It cannot make the initialisations *identical* — `r=8` and `r=64` draw different-shaped tensors —
but it makes them draws from the same seeded stream, which is as close as these axes allow.

### Two measurements that are easy to get subtly wrong

`time.perf_counter()` brackets `trainer.train()` **and nothing else**, so tokenisation, model
construction and scoring never leak into the training-time column.

`reset_peak_memory_stats()` resets the peak to the **current** allocation, not to zero. Every peak
below therefore includes the ~2.5 GB resident base model — which is the number you want, since it
is what the configuration actually costs to train, but it is not the adapter's own footprint.

In [ ]:
from transformers import Trainer, TrainingArguments


def run_arm(a):
    """Train one LoRA configuration on the merged stage-1 base, score it, hand the base back."""
    global ACTIVE_PEFT
    print(f"\n{'=' * 100}")
    print(f"{a['name']}   r={a['r']}  alpha={a['alpha']} (a/r={a['alpha']/a['r']:g})  "
          f"targets={len(a['targets'])}  lr={a['lr']:.0e}  seed={a['seed']}"
          f"{'   — ' + a['note'] if a['note'] else ''}")
    print("=" * 100)

    resident = torch.cuda.memory_allocated() / 1e9
    assert resident < 4.0, (
        f"{resident:.2f} GB still allocated before this arm starts — the previous arm leaked. "
        "Restart the runtime and re-run the sweep cell; completed arms are checkpointed.")

    model = fresh_base()
    torch.cuda.reset_peak_memory_stats()

    # See the markdown above: without this, arm k's LoRA init depends on arm k-1's training.
    transformers.set_seed(a["seed"])

    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=a["r"], lora_alpha=a["alpha"],
        lora_dropout=0.05, bias="none", target_modules=list(a["targets"])))
    ACTIVE_PEFT = model

    # Same fp16 gotcha as notebooks 01-03: trainable params must be fp32 or the grad scaler
    # refuses to unscale them. Kept on bf16 hardware too, so the arms stay comparable across GPUs
    # in the one respect we control.
    for _, param in model.named_parameters():
        if param.requires_grad and param.dtype != torch.float32:
            param.data = param.data.float()
    model.config.use_cache = False

    # Verify peft applied the scaling we asked for. Probe q_proj, not down_proj: the MLP
    # projections are unwrapped in the attn-only arm and this would raise.
    scaling = model.base_model.model.model.layers[8].self_attn.q_proj.scaling["default"]
    assert abs(scaling - a["alpha"] / a["r"]) < 1e-9, \
        f"peft applied scaling {scaling}, expected alpha/r = {a['alpha'] / a['r']}"

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    args = TrainingArguments(
        output_dir=f"/content/outputs/03a-ablation/{a['name']}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=2,      # NOT tuned to the GPU — see section 8
        per_device_eval_batch_size=2,       # changing this changes the reported eval_loss
        gradient_accumulation_steps=4,
        learning_rate=a["lr"],
        lr_scheduler_type="cosine",
        warmup_steps=WARMUP_STEPS,
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not SUPPORTS_BF16,
        bf16=SUPPORTS_BF16,
        logging_steps=LOGGING_STEPS,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        seed=a["seed"],                     # also fixes data order: data_seed falls back to it
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      eval_dataset=eval_ds, data_collator=collate)

    t0 = time.perf_counter()
    trainer.train()
    train_seconds = time.perf_counter() - t0
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    hist = trainer.state.log_history
    eval_curve = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    train_curve = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]

    t0 = time.perf_counter()
    scored = score(model, score_records, a["name"])
    score_seconds = time.perf_counter() - t0

    del trainer
    fresh_base()                    # detach this arm's adapter and restore the weights
    gc.collect(); torch.cuda.empty_cache()

    out = {
        "config": {k: a[k] for k in ("r", "alpha", "targets", "lr", "seed", "axis")},
        "accuracy": scored["accuracy"], "parse_rate": scored["parse_rate"],
        "preds": scored["preds"],   # texts deliberately NOT stored: ~9x the JSON, pushed to Drive
        "eval_loss": eval_curve[-1][1] if eval_curve else None,
        "eval_loss_min": min(p[1] for p in eval_curve) if eval_curve else None,
        "eval_curve": eval_curve, "train_curve": train_curve,
        "trainable": trainable, "trainable_pct": 100 * trainable / total,
        "peak_vram_gb": peak_gb,
        "train_seconds": train_seconds, "score_seconds": score_seconds,
    }
    print(f"  accuracy {out['accuracy']:.1%}   parse {out['parse_rate']:.1%}   "
          f"eval_loss {out['eval_loss']:.4f}   trainable {trainable/1e6:.1f}M "
          f"({out['trainable_pct']:.2f}%)   peak {peak_gb:.2f} GB   "
          f"train {train_seconds/60:.1f} min   score {score_seconds:.0f} s")

    if eval_curve and out["eval_loss"] > 1.5 * eval_curve[0][1]:
        print("  WARNING: eval loss ended >1.5x its first value — this arm diverged. Read its "
              "accuracy as a failure mode, not as a measurement.")
    elif eval_curve and out["eval_loss"] > 1.02 * out["eval_loss_min"]:
        print(f"  note: eval loss rose {out['eval_loss'] / out['eval_loss_min'] - 1:.1%} above its "
              f"minimum ({out['eval_loss_min']:.4f}) — this arm is past its best epoch.")
    if out["parse_rate"] < 0.9:
        print(f"  WARNING: parse rate {out['parse_rate']:.1%} — this arm's accuracy is low mostly "
              "because it stopped emitting a parseable 'Answer:' line.")
    return out

## 11. Checkpointing, because ninety minutes is long enough to lose

Every arm is written to disk and to Drive the moment it finishes. Re-running the sweep cell after a
disconnect picks up exactly where it stopped.

Resumption is **gated** on the things that would make old arms incomparable — GPU, compute
capability, dtype, AMP mode, base model, and every data/schedule constant. Change any of them and
the old file is ignored rather than silently mixed in. Library versions are *recorded* but not
gated: a mid-week Colab bump should print a warning, not throw away eleven completed arms.

In [ ]:
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "capability": ".".join(str(x) for x in torch.cuda.get_device_capability(0)),
    "compute_dtype": str(DTYPE).replace("torch.", ""),
    "amp": "bf16" if SUPPORTS_BF16 else "fp16",
    "model": MODEL_ID,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
}
GATED = ["gpu", "capability", "compute_dtype", "amp", "model"]
RUN_CONFIG = {k: ENV[k] for k in GATED}
RUN_CONFIG.update({
    "quick": QUICK, "epochs": EPOCHS, "n_train": N_TRAIN, "n_eval": N_EVAL,
    "warmup_steps": WARMUP_STEPS, "max_len": MAX_LEN, "seed": SEED,
    "eval_fraction": EVAL_FRACTION, "stage1": STAGE1_DIR is not None,
    "micro_batch": 2, "grad_accum": 4, "eval_batch": 2, "score_batch": 8,
})

RESULTS_NAME = f"03a-ablation-results{'-quick' if QUICK else ''}.json"
RESULTS_LOCAL = Path("/content/outputs") / RESULTS_NAME
RESULTS_DRIVE = Path("/content/drive/MyDrive/finetuning-demo") / RESULTS_NAME


def load_results():
    """Drive first: /content is what gets wiped on a disconnect."""
    for path in (RESULTS_DRIVE, RESULTS_LOCAL):
        if not path.exists():
            continue
        blob = json.loads(path.read_text())
        if blob.get("run_config") != RUN_CONFIG:
            differs = sorted(k for k in RUN_CONFIG
                             if blob.get("run_config", {}).get(k) != RUN_CONFIG[k])
            print(f"Ignoring {path} — written under a different run config (differs on {differs}).")
            continue
        if blob.get("env") != ENV:
            print(f"WARNING: {path} was written with different library versions. Resuming anyway.")
        print(f"Resuming from {path}: {len(blob['results'])} arm(s) already done.")
        return blob["results"]
    return {}


def save_results(results):
    blob = json.dumps({"run_config": RUN_CONFIG, "env": ENV, "results": results}, indent=1)
    RESULTS_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_LOCAL.write_text(blob)
    if Path("/content/drive/MyDrive").exists():
        try:
            RESULTS_DRIVE.parent.mkdir(parents=True, exist_ok=True)
            RESULTS_DRIVE.write_text(blob)
        except OSError as e:      # a flaky Drive mount must not kill an 80-minute run
            print(f"  (Drive write failed: {e} — the local copy is still current)")


results = load_results()
print(f"\n{ENV['gpu']}  ·  {ENV['amp']}  ·  torch {ENV['torch']}  ·  "
      f"transformers {ENV['transformers']}  ·  peft {ENV['peft']}")
print(f"results file: {RESULTS_LOCAL.name}")

## 12. Run blocks A and B

Safe to re-run: completed arms are skipped, and an arm whose stored config no longer matches its
name is re-run rather than trusted.

In [ ]:
CFG_KEYS = ("r", "alpha", "targets", "lr", "seed", "axis")


def run_sweep(arms):
    t_start = time.perf_counter()
    todo = [a for a in arms if results.get(a["name"], {}).get("config") != {k: a[k] for k in CFG_KEYS}]
    print(f"{len(arms) - len(todo)} of {len(arms)} arm(s) already done; running {len(todo)}.\n")
    for i, a in enumerate(arms, 1):
        want = {k: a[k] for k in CFG_KEYS}
        prior = results.get(a["name"])
        if prior and prior["config"] == want:
            print(f"[{i:2d}/{len(arms)}] skip {a['name']:16s}  acc {prior['accuracy']:.1%}")
            continue
        if prior:
            print(f"[{i:2d}/{len(arms)}] {a['name']} was run under a different config — re-running")
        results[a["name"]] = run_arm(a)
        save_results(results)
        done = sum(1 for x in arms if x["name"] in results)
        remaining = sum(1 for x in arms[i:] if x["name"] not in results)
        per = (time.perf_counter() - t_start) / max(1, i)
        print(f"  [{done}/{len(arms)}] checkpointed  |  ~{per * remaining / 60:.0f} min left")
    print(f"\nDone in {(time.perf_counter() - t_start) / 60:.1f} min.")


run_sweep(ARMS)

## 13. Block C — the winner's curse

Two of the arms above scored highest. The question this block asks is not *"how good are they?"*
but *"were they actually better, or did they just get a good seed?"*

With thirteen `seed=42` arms and a noise floor near 3.5 points, picking the top two is very largely
picking the two luckiest draws. **The expected outcome of this block is that both winners
regress** — that is what selecting on a noisy metric does, and it is the reason a leaderboard built
from a single run of each configuration is worth so much less than it looks.

So: read the predicted regression, then run the cell. If they regress, the notebook has
demonstrated the winner's curse on its own data. If they don't, that is genuine evidence the
ranking meant something.

The third arm is a **second replicate of the baseline**, at seed 44. `baseline-s43` already exists
in Block A, so re-running the baseline at 43 would buy nothing — whereas seed 44 raises the number
of paired seed-deltas from one to four. One delta is an anecdote; four is a floor.

In [ ]:
RUN_REPLICATION = True

BLOCK_C = []
if RUN_REPLICATION:
    ranked = sorted((a for a in ARMS if a["name"] in results and a["seed"] == BASE_SEED),
                    key=lambda a: (-results[a["name"]]["accuracy"], a["name"]))   # total order
    print("Ranking by accuracy — this is exactly the selection the block is about to test:")
    for a in ranked[:4]:
        print(f"    {a['name']:16s} {results[a['name']]['accuracy']:.1%}")

    for src, seed in [(ranked[0], REPLICATE_SEED), (ranked[1], REPLICATE_SEED),
                      (BLOCK_A[0], SECOND_REPLICATE_SEED)]:
        cand = arm(f"rep-s{seed}-{src['name']}", "noise", r=src["r"], alpha=src["alpha"],
                   targets=src["targets"], lr=src["lr"], seed=seed,
                   note=f"seed replicate of {src['name']}")
        if train_key(cand) in _seen:
            print(f"    (skipping {cand['name']}: an identical run is already in the sweep)")
            continue
        _seen.add(train_key(cand))
        BLOCK_C.append(cand)

    print(f"\nBlock C: {len(BLOCK_C)} replicate arm(s) — "
          f"~{len(BLOCK_C) * EST_MIN_PER_ARM:.0f} min")

In [ ]:
if BLOCK_C:
    run_sweep(BLOCK_C)
else:
    print("Block C is empty — nothing to replicate.")

## 14. The noise floor

Every pair of runs below shares a configuration **exactly** and differs only in the random seed. So
whatever separates them is not an effect — it is the measurement error of this setup, and it is the
yardstick every other number in this notebook has to clear.

In [ ]:
def mcnemar_p(pa, pb, gold):
    """Exact two-sided binomial on the discordant records only — notebook 03's test, reused.

    Both systems answered the SAME records, so the information about their difference lives
    entirely in the records where they disagree. No scipy: it is three lines.
    """
    b = sum(1 for x, y, g in zip(pa, pb, gold) if x == g and y != g)
    c = sum(1 for x, y, g in zip(pa, pb, gold) if x != g and y == g)
    n = b + c
    if n == 0:
        return 1.0, b, c
    lo = min(b, c)
    return min(1.0, 2 * sum(math.comb(n, k) for k in range(lo + 1)) / 2 ** n), b, c


def cfg_without_seed(c):
    return (c["r"], c["alpha"], tuple(c["targets"]), c["lr"])


groups = {}
for name, r in results.items():
    groups.setdefault(cfg_without_seed(r["config"]), []).append((r["config"]["seed"], name, r))

print("Seed replicates — same configuration, different `seed`, nothing else.\n")
print(f"{'configuration':18s}{'seeds':>10s}{'Δaccuracy':>12s}{'Δeval loss':>13s}{'McNemar p':>12s}")
print("-" * 65)

acc_deltas, loss_deltas, rel = [], [], []
for cfg, runs in sorted(groups.items(), key=lambda kv: str(kv[0])):
    runs.sort()
    for i in range(len(runs)):
        for j in range(i + 1, len(runs)):
            (s1, n1, r1), (s2, n2, r2) = runs[i], runs[j]
            da = r2["accuracy"] - r1["accuracy"]
            dl = r2["eval_loss"] - r1["eval_loss"]
            p, _, _ = mcnemar_p(r1["preds"], r2["preds"], gold_scored)
            acc_deltas.append(abs(da)); loss_deltas.append(abs(dl))
            if r1["accuracy"] and r1["eval_loss"]:
                rel.append((abs(da) / r1["accuracy"], abs(dl) / r1["eval_loss"]))
            print(f"{n1[:18]:18s}{f'{s1}/{s2}':>10s}{da:>+11.1%}{dl:>+13.4f}{p:>12.3f}")

NOISE_ACC = max(acc_deltas) if acc_deltas else 0.0
NOISE_LOSS = max(loss_deltas) if loss_deltas else 0.0
print("-" * 65)
print(f"{'noise floor: max |Δ|':28s}{NOISE_ACC:>11.1%}{NOISE_LOSS:>13.4f}"
      f"{f'n={len(acc_deltas)} pairs':>12s}")

if rel:
    ra = sum(x for x, _ in rel) / len(rel)
    rl = sum(y for _, y in rel) / len(rel)
    print(f"\nChanging nothing but the seed moved accuracy by {ra:.1%} of its value, and eval loss")
    print(f"by {rl:.1%} of its value — a factor of {ra / max(rl, 1e-9):.1f}.")

print("\nThat ratio is the whole argument for reporting both metrics. Eval loss is a mean over")
print("thousands of per-token log-likelihoods and is continuous in the weights, so it detects")
print("whether a knob did ANYTHING. Accuracy collapses each record to one bit after a greedy")
print("decode and a regex, so it is far noisier — but it is the only one of the two that says")
print("whether the knob did anything USEFUL.")
print(f"\nWith {len(acc_deltas)} replicate pair(s) this is NOT a standard deviation and must not be")
print(f"reported as one. It is a floor: an arm within {NOISE_ACC:.1%} of the baseline is")
print("indistinguishable from having changed nothing at all.")

## 15. Results

`Δacc` is against the baseline arm. `McN p` is the exact paired McNemar test on the same records.
An arm is starred only if it clears the measured noise floor **and** the paired test agrees.

Expect a column of dashes. Thirteen configurations, one small model, 800 training examples and 200
eval records is not enough to resolve most of these differences — and a table that admitted that
is worth more than one that didn't.

In [ ]:
REF = results["baseline"]
AXES = ["baseline", "noise", "targets", "lr", "rank", "scaling", "rank x lr"]
ALL_ARMS = ARMS + BLOCK_C

print(f"{len(score_records)} held-out records   {dict(Counter(gold_scored))}   "
      f"majority class {MAJORITY_BASELINE:.1%}")
print(f"{ENV['gpu']} · {ENV['amp']} — peak VRAM is measured on THIS GPU and does not transfer "
      f"to another.\n")

head = (f"{'arm':17s}{'r':>4s}{'a/r':>5s}{'tgt':>5s}{'lr':>8s}{'acc':>8s}{'Δacc':>8s}"
        f"{'McN p':>8s}{'eval':>8s}{'Δeval':>8s}{'train%':>8s}{'GB':>6s}{'min':>6s}{'':>5s}")
print(head)
print("=" * len(head))

n_clear = 0
for axis in AXES:
    rows = [a for a in ALL_ARMS if a["axis"] == axis and a["name"] in results]
    if not rows:
        continue
    print(f"-- {axis} " + "-" * (len(head) - len(axis) - 4))
    for a in rows:
        r = results[a["name"]]
        c = r["config"]
        p, _, _ = mcnemar_p(REF["preds"], r["preds"], gold_scored)
        dacc = r["accuracy"] - REF["accuracy"]
        dloss = r["eval_loss"] - REF["eval_loss"]
        if a["name"] == "baseline":
            flag = "ref"
        elif abs(dacc) > NOISE_ACC and p < 0.05:
            flag = "*"; n_clear += 1
        else:
            flag = "-"
        print(f"{a['name']:17s}{c['r']:>4d}{c['alpha']/c['r']:>5g}{len(c['targets']):>5d}"
              f"{c['lr']:>8.0e}{r['accuracy']:>8.1%}{dacc:>+8.1%}{p:>8.2f}"
              f"{r['eval_loss']:>8.3f}{dloss:>+8.3f}{r['trainable_pct']:>7.2f}%"
              f"{r['peak_vram_gb']:>6.2f}{r['train_seconds']/60:>6.1f}{flag:>5s}")

print("=" * len(head))
print(f"*  |Δaccuracy| exceeds the measured noise floor ({NOISE_ACC:.1%}) AND McNemar p < 0.05")
print(f"-  indistinguishable from changing nothing")
print(f"\n{n_clear} of {len(ALL_ARMS) - 1} arms cleared the noise floor.")
print(f"total GPU time: {sum(r['train_seconds'] + r['score_seconds'] for r in results.values())/60:.0f} min")

## 16. The rank × lr grid

The one interaction the one-factor-at-a-time design structurally cannot see. Read across each row:
if the best learning rate is the same at `r=8` and at `r=64`, the two axes are independent and
Block A's conclusions about each hold generally. If it shifts, then "the best rank" is not a
question with an answer on its own — and neither is "the best learning rate".

Cells marked `^` were run as part of Block A; the other four are Block B.

In [ ]:
GRID_TGT = tuple(ALL_LINEAR)
_by_key = {train_key(a): results[a["name"]] for a in ALL_ARMS if a["name"] in results}


def grid_cell(r, lr):
    return _by_key.get((r, 2 * r, GRID_TGT, lr, BASE_SEED))


for metric, fmt, better, lower_is_better in [("accuracy", "{:>9.1%}", max, False),
                                             ("eval_loss", "{:>9.3f}", min, True)]:
    print(f"\n{metric}   (alpha/r = 2, all 7 projections, seed {BASE_SEED})")
    print(f"{'':>8s}" + "".join(f"{'lr ' + lr_tag(lr):>11s}" for lr in GRID_LR))
    for r in GRID_R:
        line = f"{'r=' + str(r):>8s}"
        for lr in GRID_LR:
            c = grid_cell(r, lr)
            line += (fmt.format(c[metric]) + ("^ " if (r, lr) in BORROWED else "  ")) if c \
                else f"{'—':>11s}"
        print(line)

    print()
    for r in GRID_R:
        vals = [(lr, grid_cell(r, lr)[metric]) for lr in GRID_LR if grid_cell(r, lr)]
        if not vals:
            continue
        best = better(vals, key=lambda t: t[1])
        spread = max(v for _, v in vals) - min(v for _, v in vals)
        note = ""
        if metric == "accuracy" and spread <= NOISE_ACC:
            note = f"   (spread {spread:.1%} <= noise floor {NOISE_ACC:.1%} — 'best' is arbitrary)"
        print(f"   r={r:<3d} best lr: {lr_tag(best[0]):>6s}   spread across the row: "
              f"{spread:.3f}{note}")

best_lrs = {r: max(((lr, grid_cell(r, lr)["accuracy"]) for lr in GRID_LR if grid_cell(r, lr)),
                   key=lambda t: t[1])[0] for r in GRID_R if any(grid_cell(r, lr) for lr in GRID_LR)}
print(f"\nbest lr by accuracy per rank: {({r: lr_tag(lr) for r, lr in best_lrs.items()})}")
print("Same value in every row => rank and learning rate are separable here, and Block A's")
print("single-axis conclusions generalise. Different values => they are not, and neither axis")
print("has a best setting independent of the other.")

## 17. Plots

The grey band on every panel is the measured noise floor around the baseline. A bar or a point
inside it has not been shown to differ from notebook 02's configuration.

In [ ]:
BLUE, RED, GREY = "#4C72B0", "#C44E52", "#999999"

block_a_names = [a["name"] for a in BLOCK_A if a["name"] in results]
ys = range(len(block_a_names))

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, key, colour, label in [(axes[0], "accuracy", BLUE, "decision accuracy"),
                               (axes[1], "eval_loss", RED, "final eval loss")]:
    vals = [results[n][key] for n in block_a_names]
    ax.barh(list(ys), vals, color=colour, alpha=0.85)
    ref = REF[key]
    ax.axvline(ref, color=GREY, ls="--", lw=1.2, label="baseline")
    band = NOISE_ACC if key == "accuracy" else NOISE_LOSS
    ax.axvspan(ref - band, ref + band, color=GREY, alpha=0.18, label="noise floor")
    if key == "accuracy":
        ax.axvline(MAJORITY_BASELINE, color="black", ls=":", lw=1.2, label="majority class")
        ax.set_xlim(0, max(max(vals), MAJORITY_BASELINE) * 1.15)
    ax.set_xlabel(label)
    ax.grid(alpha=0.3, axis="x")
    ax.legend(fontsize=8, loc="lower right")

axes[0].set_yticks(list(ys)); axes[0].set_yticklabels(block_a_names, fontsize=9)
axes[0].invert_yaxis()
fig.suptitle("Block A — one factor at a time, against notebook 02's configuration")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colours = {8: BLUE, 16: "#55A868", 64: RED}

for ax, key, label in [(axes[0], "accuracy", "decision accuracy"),
                       (axes[1], "eval_loss", "final eval loss")]:
    for r in GRID_R:
        xs = [lr for lr in GRID_LR if grid_cell(r, lr)]
        vs = [grid_cell(r, lr)[key] for lr in xs]
        if xs:
            ax.plot(xs, vs, marker="o", color=colours[r], label=f"r={r}")
    ref = REF[key]
    band = NOISE_ACC if key == "accuracy" else NOISE_LOSS
    ax.axhspan(ref - band, ref + band, color=GREY, alpha=0.18, label="noise floor")
    ax.set_xscale("log"); ax.set_xlabel("learning rate"); ax.set_ylabel(label)
    ax.set_xticks(GRID_LR); ax.set_xticklabels([lr_tag(lr) for lr in GRID_LR])
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

fig.suptitle("Block B — does the best learning rate depend on the rank?")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
pts = [(results[a["name"]]["trainable"], results[a["name"]]["accuracy"], a["name"])
       for a in ALL_ARMS if a["name"] in results]

plt.scatter([p[0] for p in pts], [p[1] for p in pts], color=BLUE, s=45, zorder=3)
for x, y, n in pts:
    plt.annotate(n, (x, y), fontsize=7, xytext=(4, 4), textcoords="offset points")

plt.axhspan(REF["accuracy"] - NOISE_ACC, REF["accuracy"] + NOISE_ACC,
            color=GREY, alpha=0.18, label="noise floor around baseline")
plt.axhline(MAJORITY_BASELINE, color="black", ls=":", lw=1.2, label="majority class")
plt.xscale("log"); plt.xlabel("trainable parameters"); plt.ylabel("decision accuracy")
plt.title("Does buying more trainable parameters buy accuracy?")
plt.grid(alpha=0.3); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

lo, hi = min(p[0] for p in pts), max(p[0] for p in pts)
print(f"parameter range: {lo/1e6:.1f}M -> {hi/1e6:.1f}M ({hi/lo:.0f}x)   "
      f"accuracy range: {min(p[1] for p in pts):.1%} -> {max(p[1] for p in pts):.1%}")

## 18. What this can and cannot tell you

### What was measured

Sixteen adapters on the same merged stage-1 base, the same 800 training records in the same order,
the same 200 held-out records, and a measured noise floor from four same-configuration seed pairs.

### The confounds, stated plainly

- **One factor at a time cannot see interactions.** Block B buys exactly one of them — rank × learning
  rate. Targets × learning rate and `alpha/r` × rank remain untested, and an arm that
  underperformed at its single tested setting might not at another.
- **`alpha/r` and `lr` are partially redundant, but not equivalent.** `alpha/r` multiplies the
  adapter's contribution in the *forward* pass; `lr` multiplies the *step* taken on `A` and `B`.
  Under plain SGD "alpha is just a learning rate" is roughly right. Under AdamW it is not, because
  Adam normalises per-parameter gradient magnitude and a uniform rescaling largely cancels in
  `m/√v`. Treat the two axes as related and hard to predict, not as duplicates.
- **The rank arms are still confounded, just not in the obvious way.** They hold `alpha/r = 2`
  fixed, which is the usual advice — but at fixed `alpha/r` the magnitude of `BA` still grows
  roughly as `√r`, because `A`'s per-entry scale is set by fan-in while `BA` sums `r` terms. peft's
  `use_rslora=True` scales by `alpha/√r` instead and is the standard correction. If the rank axis
  looks strange, that is the first thing to try.
- **Sixteen arms against one baseline is multiple comparisons.** At α=0.05 you would expect roughly
  0.75 false positives from chance alone. The honest correction (Bonferroni, α≈0.003) would make it
  nearly impossible for anything to clear at n=200 — which is itself the finding, and the reason
  Block C replicates rather than trusting the ranking.
- **Selecting the top arms and reporting their scores is biased upward.** That is what Block C
  exists to demonstrate.

### Where the hardware leaks in

Every comparison in this notebook is **internal to this notebook**, and that is deliberate. The
baseline arm is notebook 02's *configuration*, retrained here, and it is the reference the other
fifteen arms are measured against — but it will not land on notebook 02's printed number, for
reasons that have nothing to do with the sweep:

- Neither notebook seeds its LoRA initialisation before `get_peft_model` — notebook 02 is
  reproducible per top-to-bottom run, not across runs. This notebook seeds every arm, which is what
  makes the arms comparable *to each other*.
- Base weights are **bf16 on Ampere and later, fp16 on a T4** — different mantissas, so the loaded
  base and the stage-1 merge both differ before a single step is taken.
- fp16 training uses a gradient scaler that skips steps while it calibrates; bf16 does not.
- SDPA selects flash/efficient attention kernels on `sm_80` but falls back to a math implementation
  with a materialised score matrix on `sm_75`, so reduction order differs and losses diverge.
- Greedy decoding is not hardware-invariant: near-ties in the argmax resolve differently, and one
  flipped token can flip a parsed decision.
- **`peak VRAM` therefore does not transfer between GPUs.** An A100 figure says nothing about
  whether the same configuration fits a T4.

### The honest scope

800 training examples, 200 eval records, one dataset, one 1B model, one task. This notebook
demonstrates **how to run a sweep and how to read one**. It is not a table of recommended
hyperparameters, and the specific winners here should not be carried to another model or scale.

### Where to go next

| Change | Why |
|---|---|
| **More replicates per arm** | Three seeds per configuration turns the floor into a real standard deviation and lets you use a proper test instead of a threshold. The single highest-value change here. |
| **A bigger eval set** | `pqa_artificial` has 211k records. n=200 is what makes the floor 3.5 points; n=2000 would make it ~1.1. |
| **`use_rslora=True`** | Removes the `√r` scaling confound from the rank axis. |
| **Bayesian search over the joint space** | Optuna or Ax find good corners of a 4-D space in far fewer than 54 runs, and give importance estimates that one-factor-at-a-time cannot. |
| **Sweep on the stage-1 objective too** | Perplexity is continuous and far cheaper to measure than decision accuracy, so the same GPU hour resolves much smaller effects. |
| **`lora_dropout` and `weight_decay`** | Held fixed here, and both plausibly matter at 800 examples. |

See `docs/concepts.md` §3 for the claims this notebook set out to test, and notebook 02 for the
pipeline whose configuration it varies.